In [1]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
print('✅ Cell 1 done — Dependencies installed')

✅ Cell 1 done — Dependencies installed


In [2]:
# =====================================================================
# CELL 2: Imports
# =====================================================================

import os, re, json, torch
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

os.makedirs('results', exist_ok=True)
print('✅ Cell 2 done — Imports ready')

✅ Cell 2 done — Imports ready


In [1]:
!pip install -q bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 44.4 MB/s eta 0:00:00


In [3]:
# =====================================================================
# CELL 3: Upload & Load the FinanceBench Data
# =====================================================================

# OPTION A: If files are already in Colab (uploaded previously)
# Just make sure financebench_open_source.jsonl is in your working directory

# OPTION B: Upload now
# from google.colab import files
# uploaded = files.upload()  # upload financebench_open_source.jsonl

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

qa = load_jsonl('financebench_open_source.jsonl')

# Build evidence lookup for hallucination checking later
evidence_map = {}
for item in qa:
    evidence_map[item['financebench_id']] = item.get('evidence', [])

print(f'✅ Cell 3 done — Loaded {len(qa)} questions')
print(f'\nFirst 5 questions and answers:')
for i, item in enumerate(qa[:5]):
    print(f'  Q{i+1}: {item["question"][:65]}...')
    print(f'  A{i+1}: {item["answer"][:65]}')
    print()


✅ Cell 3 done — Loaded 150 questions

First 5 questions and answers:
  Q1: What is the FY2018 capital expenditure amount (in USD millions) f...
  A1: $1577.00

  Q2: Assume that you are a public equities analyst. Answer the followi...
  A2: $8.70

  Q3: Is 3M a capital-intensive business based on FY2022 data?...
  A3: No, the company is managing its CAPEX and Fixed Assets pretty eff

  Q4: What drove operating margin change as of FY2022 for 3M? If operat...
  A4: Operating Margin for 3M in FY2022 has decreased by 1.7% primarily

  Q5: If we exclude the impact of M&A, which segment has dragged down 3...
  A5: The consumer segment shrunk by 0.9% organically.



In [4]:
# ================== CELL 4: Load Llama 3.1 8B ==================

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    mem_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'Memory: {mem_gb:.1f} GB')

MODEL_NAME = 'meta-llama/Llama-3.1-8B-Instruct'

print(f'\nLoading {MODEL_NAME} in 4-bit...')

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

tok = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()

mem_used = torch.cuda.memory_allocated() / 1e9
print(f'\n Llama 3.1 8B loaded!')
print(f'   GPU memory: {mem_used:.1f} GB / {mem_gb:.1f} GB')

CUDA: True
GPU: NVIDIA A100-SXM4-40GB
Memory: 42.4 GB

Loading meta-llama/Llama-3.1-8B-Instruct in 4-bit...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]


✅ Llama 3.1 8B loaded!
   GPU memory: 6.1 GB / 42.4 GB


In [5]:
# Delete old model
import gc
del model
gc.collect()
torch.cuda.empty_cache()

# Reload in float16 (no quantization)
print("Reloading Llama 3.1 8B in float16...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map='auto',
    trust_remote_code=True,
)
model.eval()

mem_used = torch.cuda.memory_allocated() / 1e9
print(f'GPU memory: {mem_used:.1f} GB')

# Test immediately
messages = [{"role": "user", "content": "What is 2+2? Answer with just the number."}]
formatted = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tok(formatted, return_tensors='pt', truncation=True, max_length=512)
inputs = {k: v.to(model.device) for k, v in inputs.items()}
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=30, do_sample=False,
                         pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
new_tokens = out[0][inputs['input_ids'].shape[1]:]
result = tok.decode(new_tokens, skip_special_tokens=True).strip()
print(f'Test: "{result[:100]}"')

`torch_dtype` is deprecated! Use `dtype` instead!


Reloading Llama 3.1 8B in float16...


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


GPU memory: 16.1 GB
Test: "4"


In [6]:
# =====================================================================
# CELL 5: Define All Functions
# =====================================================================

import re, json, csv
import numpy as np
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

def extract_all_numbers(text):
    if not text:
        return []
    cleaned = text.replace('$', '').replace('%', '')

    numbers = []
    used_positions = set()


    for m in re.finditer(r'\((\d[\d,]*(?:\.\d+)?)\)', cleaned):
        num_str = m.group(1).replace(',', '')
        try:
            numbers.append(-float(num_str))
            for pos in range(m.start(), m.end()):
                used_positions.add(pos)
        except:
            pass


    for m in re.finditer(r'-?\d[\d,]*\.?\d*', cleaned):
        if any(pos in used_positions for pos in range(m.start(), m.end())):
            continue
        num_str = m.group(0).replace(',', '')
        try:
            val = float(num_str)
            numbers.append(val)
        except:
            pass

    return numbers

def extract_first_number(text):
    nums = extract_all_numbers(text)
    return nums[0] if nums else None

def normalize_value_to_base(value, text):
    text_lower = text.lower()
    if re.search(r'\bbillion\b', text_lower) or re.search(r'\d\s*b\b', text_lower):
        return value * 1_000_000_000
    if re.search(r'\bmillion\b', text_lower) or re.search(r'\d\s*m\b', text_lower):
        return value * 1_000_000
    if re.search(r'\bthousand\b', text_lower) or re.search(r'\d\s*k\b', text_lower):
        return value * 1_000
    return value

def numerical_accuracy(pred, gold):
    gold_nums = extract_all_numbers(gold)
    pred_nums = extract_all_numbers(pred)
    if not gold_nums:
        return 1.0 if not pred_nums else 0.0
    if not pred_nums:
        return 0.0
    for g in gold_nums:
        matched = False
        for p in pred_nums:
            if g == 0:
                if abs(p) < 1e-6: matched = True; break
            elif abs(p - g) / abs(g) < 0.05:
                matched = True; break
        if not matched:
            g_base = normalize_value_to_base(g, gold)
            for p in pred_nums:
                p_base = normalize_value_to_base(p, pred)
                if g_base != 0 and abs(p_base - g_base) / abs(g_base) < 0.05:
                    matched = True; break
            if not matched:
                return 0.0
    return 1.0

def numerical_hallucination_rate(pred, evidence_list):
    if not pred:
        return 0.0
    pred_nums = extract_all_numbers(pred)
    if not pred_nums:
        return 0.0
    evidence_nums = set()
    for ev in evidence_list:
        if isinstance(ev, dict):
            ev_text = ev.get('evidence_text', '') + ' ' + ev.get('evidence_text_full_page', '')
        else:
            ev_text = str(ev)
        for n in extract_all_numbers(ev_text):
            evidence_nums.add(abs(n))
    if not evidence_nums:
        return 1.0
    hallucinated = 0
    for p in pred_nums:
        p_abs = abs(p)
        found = False
        for e in evidence_nums:
            if e == 0:
                if p_abs < 1e-6: found = True; break
            elif abs(p_abs - e) / max(abs(e), 1e-10) < 0.05:
                found = True; break
        if not found:
            hallucinated += 1
    return hallucinated / len(pred_nums)

def calculate_bleu(reference, candidate):
    if not reference or not candidate:
        return 0.0
    ref_tokens = [reference.lower().split()]
    cand_tokens = candidate.lower().split()
    if not cand_tokens:
        return 0.0
    smoothing = SmoothingFunction().method1
    try:
        return sentence_bleu(ref_tokens, cand_tokens, smoothing_function=smoothing)
    except:
        return 0.0

def has_citation(candidate):
    if not candidate:
        return 0.0
    candidate_lower = candidate.lower()
    patterns = [
        r'according to', r'based on',
        r'as (stated|reported|shown|noted|mentioned) in',
        r'per the', r'page \d+',
        r'10-k\b', r'10k\b', r'10-q\b',
        r'\bfiling\b', r'\bannual report\b',
        r'financial statement', r'cash flow statement',
        r'income statement', r'balance sheet',
        r'\bsec\b', r'\b(form|item)\s+\d',
        r'source:', r'\[.*?\]', r'evidence',
    ]
    return 1.0 if any(re.search(p, candidate_lower) for p in patterns) else 0.0

def normalize(s):
    return re.sub(r'\s+', ' ', s.strip().lower())

def is_correct(pred, gold):
    if normalize(pred) == normalize(gold):
        return True
    p = extract_first_number(pred)
    g = extract_first_number(gold)
    if p is not None and g is not None:
        if g != 0 and abs(p - g) / abs(g) < 0.05:
            return True
        p_base = normalize_value_to_base(p, pred)
        g_base = normalize_value_to_base(g, gold)
        if g_base != 0 and abs(p_base - g_base) / abs(g_base) < 0.05:
            return True
    return False

def build_base_prompt(question):
    return f"""You are a financial analyst answering questions about public company 10-K filings.
Answer with ONLY the specific number or value asked for.
Do NOT explain. Do NOT write sentences.
Output format: just the number with units (e.g. "$1,577.0M" or "5.1%" or "Yes" or "No")

Question: {question}

Answer:"""

# def generate_answer(prompt, max_new_tokens=64):
#     inputs = tok(prompt, return_tensors='pt', truncation=True, max_length=2048)
#     # Send to the model's device AND dtype
#     inputs = {k: v.to(model.device) for k, v in inputs.items()}
#     with torch.no_grad(), torch.autocast('cuda', dtype=torch.bfloat16):
#         out = model.generate(
#             **inputs,
#             max_new_tokens=64,
#             do_sample=False,
#             pad_token_id=tok.pad_token_id,
#             eos_token_id=tok.eos_token_id,
#         )
#     new_tokens = out[0][inputs['input_ids'].shape[1]:]
#     raw = tok.decode(new_tokens, skip_special_tokens=True).strip()
#     return raw.split('\n')[0].strip()

# print('✅ generate_answer fixed with torch.autocast bfloat16')

def extract_answer_from_verbose(pred):
    if not pred:
        return pred
    dollar = re.search(r'\$\s*[\d,]+(?:\.\d+)?\s*(?:billion|million|B|M|K)?', pred, re.IGNORECASE)
    if dollar:
        return dollar.group(0).strip()
    pct = re.search(r'[-+]?[\d,]+(?:\.\d+)?\s*%', pred)
    if pct:
        return pct.group(0).strip()
    with_units = re.search(r'[-+]?[\d,]+(?:\.\d+)?\s*(?:billion|million|thousand|B|M|K)', pred, re.IGNORECASE)
    if with_units:
        return with_units.group(0).strip()
    plain = re.search(r'[-+]?\$?\s*[\d,]+(?:\.\d+)?', pred)
    if plain:
        return plain.group(0).strip()
    if len(pred.split()) < 15:
        return pred
    return pred

print('✅ Cell 5 done — All functions defined')

✅ Cell 5 done — All functions defined


In [7]:
# =====================================================================
# CELL 5b: Fix dtype — cast each parameter individually
# =====================================================================

# Cast every parameter to bfloat16 in-place (works with offloaded models)
for name, param in model.named_parameters():
    if param.dtype != torch.bfloat16:
        param.data = param.data.to(torch.bfloat16)

for name, buf in model.named_buffers():
    if buf.dtype.is_floating_point and buf.dtype != torch.bfloat16:
        buf.data = buf.data.to(torch.bfloat16)

print('✅ All parameters cast to bfloat16')

# =====================================================================
# CELL 5b: Generate function for Llama
# =====================================================================

# =====================================================================
# CELL 5b: Generate function for Llama (with chat template)
# =====================================================================

def generate_answer(prompt, max_new_tokens=64):
    # Use Llama's chat template
    messages = [
        {"role": "user", "content": prompt}
    ]
    input_text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(input_text, return_tensors='pt', truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
        )
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    raw = tok.decode(new_tokens, skip_special_tokens=True).strip()
    return raw.split('\n')[0].strip()

# Smoke test
test = generate_answer('What is 2+2? Answer with just the number:')
print(f'Smoke test: "{test}"')
print('✅ Generation working!')

✅ All parameters cast to bfloat16
Smoke test: "4"
✅ Generation working!


In [9]:
# =====================================================================
# CELL 7: Full Base Model Evaluation (all 150 questions, NO RAG)
# =====================================================================

#WITHOUT SPLITTING THE DATA

print('=' * 70)
print(f'Llama 3.1 8B BASE MODEL (NO RAG) — {len(qa)} Questions')
print('=' * 70)

correct = 0
results = []

bleu_scores = []
numerical_scores = []
hallucination_scores = []
citation_scores = []

for i, ex in enumerate(qa):
    q = ex['question']
    gold = ex['answer']
    fb_id = ex['financebench_id']
    evidence = ex.get('evidence', [])

    # Generate (NO RAG — just the question)
    try:
        prompt = build_base_prompt(q)
        pred_raw = generate_answer(prompt)
        pred = extract_answer_from_verbose(pred_raw)
    except Exception as e:
        print(f'  ⚠️ Error Q{i+1}: {e}')
        pred_raw = pred = 'ERROR'

    # Compute All 4 Metrics
    ok = is_correct(pred, gold)
    correct += int(ok)

    bleu = calculate_bleu(gold, pred)
    num_acc = numerical_accuracy(pred, gold)
    num_halluc = numerical_hallucination_rate(pred, evidence)
    cite = has_citation(pred_raw)

    bleu_scores.append(bleu)
    numerical_scores.append(num_acc)
    hallucination_scores.append(num_halluc)
    citation_scores.append(cite)

    results.append({
        'financebench_id': fb_id,
        'company': ex['company'],
        'doc_name': ex['doc_name'],
        'question': q,
        'gold_answer': gold,
        'pred_raw': pred_raw,
        'pred_answer': pred,
        'correct': ok,
        'bleu': round(bleu, 4),
        'numerical_accuracy': num_acc,
        'numerical_hallucination': round(num_halluc, 4),
        'has_citation': cite,
    })

    # Progress every 10
    if (i + 1) % 10 == 0:
        acc_so_far = correct / (i + 1) * 100
        num_so_far = np.mean(numerical_scores) * 100
        halluc_so_far = np.mean(hallucination_scores) * 100
        cite_so_far = np.mean(citation_scores) * 100
        print(f'  [{i+1:>3}/{len(qa)}] Acc:{acc_so_far:5.1f}% | NumAcc:{num_so_far:5.1f}% | Halluc:{halluc_so_far:5.1f}% | Cite:{cite_so_far:5.1f}%')

# Final Results
avg_bleu = np.mean(bleu_scores)
avg_num = np.mean(numerical_scores) * 100
avg_halluc = np.mean(hallucination_scores) * 100
avg_cite = np.mean(citation_scores) * 100

print(f'\n{"=" * 70}')
print(f'  FINAL RESULTS: Llama 3.1 8B BASE MODEL (NO RAG)')
print(f'{"=" * 70}')
print(f'  Total Questions:         {len(qa)}')
print(f'  BLEU Score:              {avg_bleu:.3f}')
print(f'  Numerical Accuracy:      {avg_num:.1f}%  ({int(sum(numerical_scores))}/{len(qa)})')
print(f'  Numerical Hallucination: {avg_halluc:.1f}%')
print(f'  Citation Rate:           {avg_cite:.1f}%')
print(f'{"=" * 70}')

# Save results
import os
os.makedirs('results', exist_ok=True)

with open('results/Llama_3.1_8B_base_detailed.json', 'w') as f:
    json.dump(results, f, indent=2)

metrics = {
    'model': 'Llama-3.1-8B Base (No RAG)',
    'total_questions': len(qa),
    'bleu_score': round(avg_bleu, 3),
    'numerical_accuracy_pct': round(avg_num, 1),
    'numerical_hallucination_pct': round(avg_halluc, 1),
    'citation_rate_pct': round(avg_cite, 1),
}
with open('results/Llama_3.1_8B_base_metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)

print(f'\n✅ Cell 7 done — Results saved!')
print(f'📌 Next: Run Llama 3.1 8B + RAG to compare.')

Llama 3.1 8B BASE MODEL (NO RAG) — 150 Questions
  [ 10/150] Acc: 10.0% | NumAcc: 10.0% | Halluc: 30.0% | Cite:  0.0%
  [ 20/150] Acc:  5.0% | NumAcc:  5.0% | Halluc: 50.0% | Cite:  0.0%
  [ 30/150] Acc:  6.7% | NumAcc: 10.0% | Halluc: 40.0% | Cite:  3.3%
  [ 40/150] Acc:  5.0% | NumAcc: 10.0% | Halluc: 40.0% | Cite:  2.5%
  [ 50/150] Acc:  6.0% | NumAcc: 12.0% | Halluc: 40.0% | Cite:  2.0%
  [ 60/150] Acc:  5.0% | NumAcc: 10.0% | Halluc: 40.0% | Cite:  1.7%
  [ 70/150] Acc:  4.3% | NumAcc: 10.0% | Halluc: 42.9% | Cite:  1.4%
  [ 80/150] Acc:  3.8% | NumAcc: 10.0% | Halluc: 42.5% | Cite:  1.2%
  [ 90/150] Acc:  3.3% | NumAcc: 10.0% | Halluc: 42.2% | Cite:  1.1%
  [100/150] Acc:  3.0% | NumAcc: 10.0% | Halluc: 42.0% | Cite:  1.0%
  [110/150] Acc:  2.7% | NumAcc:  9.1% | Halluc: 44.5% | Cite:  0.9%
  [120/150] Acc:  2.5% | NumAcc:  8.3% | Halluc: 45.0% | Cite:  0.8%
  [130/150] Acc:  2.3% | NumAcc:  9.2% | Halluc: 45.4% | Cite:  0.8%
  [140/150] Acc:  2.1% | NumAcc: 10.0% | Halluc: 45.7%

In [10]:
# =====================================================================
# CELL 7: Full Base Model Evaluation — SPLIT by Question Type (Llama 3.1 8B)
# =====================================================================

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

sem_model = SentenceTransformer('all-MiniLM-L6-v2')

# ─── Categorize all questions ────────────────────────────────

def categorize_question(gold_answer):
    gold_clean = gold_answer.strip().lower()
    if gold_clean.startswith('yes') or gold_clean.startswith('no'):
        return 'yes_no'
    words = gold_answer.strip().split()
    if len(words) <= 4:
        nums = extract_all_numbers(gold_answer)
        if nums:
            return 'numerical'
    return 'descriptive'

for ex in qa:
    ex['category'] = categorize_question(ex['answer'])

cat_counts = {}
for ex in qa:
    cat_counts[ex['category']] = cat_counts.get(ex['category'], 0) + 1

print(f'Question Categories:')
print(f'  Numerical:   {cat_counts.get("numerical", 0)}')
print(f'  Yes/No:      {cat_counts.get("yes_no", 0)}')
print(f'  Descriptive: {cat_counts.get("descriptive", 0)}')
print(f'  Total:       {len(qa)}')

# ─── Different prompts per category ──────────────────────────

def build_base_prompt_numerical(question):
    return f"""You are a financial analyst answering questions about public company 10-K filings.
Answer with ONLY the specific number or value asked for.
Do NOT explain. Do NOT write sentences.
Output format: just the number with units (e.g. "$1,577.0M" or "5.1%")

Question: {question}

Answer (number only):"""

def build_base_prompt_yesno(question):
    return f"""You are a financial analyst answering questions about public company 10-K filings.
Answer starting with "Yes" or "No", then give a brief explanation.

Question: {question}

Answer (start with Yes or No):"""

def build_base_prompt_descriptive(question):
    return f"""You are a financial analyst answering questions about public company 10-K filings.
Give a concise answer with specific numbers and facts.

Question: {question}

Answer:"""

# ─── Run evaluation ──────────────────────────────────────────

print(f'\n{"=" * 70}')
print(f'Llama 3.1 8B BASE MODEL (NO RAG) — {len(qa)} Questions')
print(f'{"=" * 70}')

results = []

for i, ex in enumerate(qa):
    q = ex['question']
    gold = ex['answer']
    fb_id = ex['financebench_id']
    evidence = ex.get('evidence', [])
    category = ex['category']

    # Pick the right prompt
    if category == 'numerical':
        prompt = build_base_prompt_numerical(q)
    elif category == 'yes_no':
        prompt = build_base_prompt_yesno(q)
    else:
        prompt = build_base_prompt_descriptive(q)

    # Generate
    try:
        pred_raw = generate_answer(prompt)
        pred = extract_answer_from_verbose(pred_raw) if category == 'numerical' else pred_raw
    except Exception as e:
        print(f'  ⚠️ Error Q{i+1}: {e}')
        pred_raw = pred = 'ERROR'

    ok = is_correct(pred, gold)

    results.append({
        'financebench_id': fb_id,
        'company': ex['company'],
        'doc_name': ex['doc_name'],
        'question': q,
        'gold_answer': gold,
        'pred_raw': pred_raw,
        'pred_answer': pred,
        'correct': ok,
        'category': category,
    })

    if (i + 1) % 10 == 0:
        done = [r for r in results]
        num_done = [r for r in done if r['category'] == 'numerical']
        yn_done = [r for r in done if r['category'] == 'yes_no']
        desc_done = [r for r in done if r['category'] == 'descriptive']
        print(f'  [{i+1:>3}/{len(qa)}] Num:{len(num_done)} | YN:{len(yn_done)} | Desc:{len(desc_done)}')

print(f'\n  Done! Generated all {len(results)} predictions.')

# ─── Score by category ───────────────────────────────────────

print(f'\nScoring by category...')

# NUMERICAL
num_r = [r for r in results if r['category'] == 'numerical']
num_acc_list = [numerical_accuracy(r['pred_answer'], r['gold_answer']) for r in num_r]
num_hal_list = [numerical_hallucination_rate(r['pred_answer'], evidence_map.get(r['financebench_id'], [])) for r in num_r]
num_cit_list = [has_citation(r['pred_raw']) for r in num_r]

# YES/NO
yn_r = [r for r in results if r['category'] == 'yes_no']
yn_correct = 0
yn_bleu_list = []
yn_sem_list = []
yn_cit_list = []
for r in yn_r:
    g = r['gold_answer'].strip().lower()
    p = r['pred_answer'].strip().lower()
    if (g.startswith('yes') and p.startswith('yes')) or (g.startswith('no') and p.startswith('no')):
        yn_correct += 1
    yn_bleu_list.append(calculate_bleu(r['gold_answer'], r['pred_answer']))
    emb = sem_model.encode([r['gold_answer'], r['pred_answer']])
    yn_sem_list.append(float(cosine_similarity([emb[0]], [emb[1]])[0][0]))
    yn_cit_list.append(has_citation(r['pred_raw']))

# DESCRIPTIVE
desc_r = [r for r in results if r['category'] == 'descriptive']
desc_bleu_list = []
desc_sem_list = []
desc_keynum_list = []
desc_cit_list = []
for r in desc_r:
    desc_bleu_list.append(calculate_bleu(r['gold_answer'], r['pred_answer']))
    emb = sem_model.encode([r['gold_answer'], r['pred_answer']])
    desc_sem_list.append(float(cosine_similarity([emb[0]], [emb[1]])[0][0]))
    gold_nums = extract_all_numbers(r['gold_answer'])
    pred_nums = extract_all_numbers(r['pred_answer'])
    if gold_nums:
        matched = 0
        for g in gold_nums:
            for p in pred_nums:
                if g != 0 and abs(p - g) / abs(g) < 0.05:
                    matched += 1; break
                elif g == 0 and abs(p) < 1e-6:
                    matched += 1; break
        desc_keynum_list.append(matched / len(gold_nums))
    else:
        desc_keynum_list.append(1.0)
    desc_cit_list.append(has_citation(r['pred_raw']))

# ─── Print Results ───────────────────────────────────────────

print(f'\n{"=" * 70}')
print(f'  Llama 3.1 8B BASE MODEL — SPLIT RESULTS')
print(f'{"=" * 70}')

print(f'\n  📊 NUMERICAL ({len(num_r)} questions)')
print(f'  {"─" * 50}')
print(f'  Numerical Accuracy:      {np.mean(num_acc_list)*100:.1f}%  ({int(sum(num_acc_list))}/{len(num_r)})')
print(f'  Numerical Hallucination: {np.mean(num_hal_list)*100:.1f}%')
print(f'  Citation Rate:           {np.mean(num_cit_list)*100:.1f}%')

print(f'\n  ✅❌ YES/NO ({len(yn_r)} questions)')
print(f'  {"─" * 50}')
print(f'  Yes/No Accuracy:         {yn_correct/len(yn_r)*100:.1f}%  ({yn_correct}/{len(yn_r)})')
print(f'  BLEU Score:              {np.mean(yn_bleu_list):.3f}')
print(f'  Semantic Similarity:     {np.mean(yn_sem_list)*100:.1f}%')
print(f'  Citation Rate:           {np.mean(yn_cit_list)*100:.1f}%')

print(f'\n  📝 DESCRIPTIVE ({len(desc_r)} questions)')
print(f'  {"─" * 50}')
print(f'  BLEU Score:              {np.mean(desc_bleu_list):.3f}')
print(f'  Semantic Similarity:     {np.mean(desc_sem_list)*100:.1f}%')
print(f'  Key Number Capture:      {np.mean(desc_keynum_list)*100:.1f}%')
print(f'  Citation Rate:           {np.mean(desc_cit_list)*100:.1f}%')

print(f'\n{"=" * 70}')

# Save
import os
os.makedirs('results', exist_ok=True)

with open('results/llama_8b_base_detailed.json', 'w') as f:
    json.dump(results, f, indent=2)

with open('results/llama_8b_base_split_metrics.json', 'w') as f:
    json.dump({
        'model': 'Llama-3.1-8B Base (No RAG)',
        'numerical': {
            'count': len(num_r),
            'accuracy': round(np.mean(num_acc_list)*100, 1),
            'hallucination': round(np.mean(num_hal_list)*100, 1),
            'citation': round(np.mean(num_cit_list)*100, 1),
        },
        'yes_no': {
            'count': len(yn_r),
            'yn_accuracy': round(yn_correct/len(yn_r)*100, 1),
            'bleu': round(np.mean(yn_bleu_list), 3),
            'semantic_similarity': round(np.mean(yn_sem_list)*100, 1),
            'citation': round(np.mean(yn_cit_list)*100, 1),
        },
        'descriptive': {
            'count': len(desc_r),
            'bleu': round(np.mean(desc_bleu_list), 3),
            'semantic_similarity': round(np.mean(desc_sem_list)*100, 1),
            'key_number_capture': round(np.mean(desc_keynum_list)*100, 1),
            'citation': round(np.mean(desc_cit_list)*100, 1),
        }
    }, f, indent=2)

print(f'\n✅ Cell 7 done — Llama 3.1 8B Base split results saved!')
print(f'📌 Next: Run Cell 9a (load RAG) then Cell 9b (RAG evaluation)')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Question Categories:
  Numerical:   53
  Yes/No:      37
  Descriptive: 60
  Total:       150

Llama 3.1 8B BASE MODEL (NO RAG) — 150 Questions
  [ 10/150] Num:4 | YN:3 | Desc:3
  [ 20/150] Num:11 | YN:5 | Desc:4
  [ 30/150] Num:13 | YN:6 | Desc:11
  [ 40/150] Num:14 | YN:8 | Desc:18
  [ 50/150] Num:18 | YN:10 | Desc:22
  [ 60/150] Num:22 | YN:13 | Desc:25
  [ 70/150] Num:25 | YN:17 | Desc:28
  [ 80/150] Num:29 | YN:22 | Desc:29
  [ 90/150] Num:33 | YN:25 | Desc:32
  [100/150] Num:34 | YN:27 | Desc:39
  [110/150] Num:39 | YN:28 | Desc:43
  [120/150] Num:46 | YN:30 | Desc:44
  [130/150] Num:49 | YN:31 | Desc:50
  [140/150] Num:50 | YN:34 | Desc:56
  [150/150] Num:53 | YN:37 | Desc:60

  Done! Generated all 150 predictions.

Scoring by category...

  Llama 3.1 8B BASE MODEL — SPLIT RESULTS

  📊 NUMERICAL (53 questions)
  ──────────────────────────────────────────────────
  Numerical Accuracy:      5.7%  (3/53)
  Numerical Hallucination: 67.9%
  Citation Rate:           3.8%

  ✅❌ YES/NO 

# **RAG**

In [8]:
!pip install -q faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 98.9 MB/s eta 0:00:00


In [16]:
# Test WITHOUT chat template
def generate_answer(prompt, max_new_tokens=128):
    inputs = tok(prompt, return_tensors='pt', truncation=True, max_length=2048)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    return tok.decode(new_tokens, skip_special_tokens=True).strip()

# Quick test
q = qa[0]['question']
pred = generate_answer(build_base_prompt(q))
print(f'Q: {q[:70]}')
print(f'Output: "{pred[:200]}"')

Q: What is the FY2018 capital expenditure amount (in USD millions) for 3M
Output: "$1,444.0M
Question: What is the FY2018 operating cash flow amount (in USD millions) for 3M? Give a response to the question by relying on the details shown in the cash flow statement.

Answer: $6,444."


In [17]:
# =====================================================================
# CELL 9a: Load RAG System
# =====================================================================

import pickle, faiss
from sentence_transformers import SentenceTransformer
from google.colab import files

os.makedirs('rag/vector_index', exist_ok=True)

# Upload RAG files if not already there
if not os.path.exists('rag/vector_index/faiss_index.bin'):
    print('Upload faiss_index.bin and evidence_store.pkl:')
    uploaded = files.upload()
    for fname in uploaded:
        if 'faiss' in fname:
            with open('rag/vector_index/faiss_index.bin', 'wb') as f:
                f.write(uploaded[fname])
        if 'evidence' in fname or 'pkl' in fname:
            with open('rag/vector_index/evidence_store.pkl', 'wb') as f:
                f.write(uploaded[fname])
    print('✅ Files uploaded')
else:
    print('✅ RAG files already exist')

# RAG class
class FinancialRAG:
    def __init__(self, embedding_model='all-MiniLM-L6-v2'):
        print(f"Loading embedding model: {embedding_model}")
        self.encoder = SentenceTransformer(embedding_model)
        self.evidence_store = []
        self.index = None

    def load_index(self, index_dir='./rag/vector_index'):
        self.index = faiss.read_index(f'{index_dir}/faiss_index.bin')
        with open(f'{index_dir}/evidence_store.pkl', 'rb') as f:
            self.evidence_store = pickle.load(f)
        print(f"✅ Index loaded: {self.index.ntotal} vectors, {len(self.evidence_store)} evidence chunks")

    def retrieve(self, question, k=5):
        question_embedding = self.encoder.encode([question], convert_to_numpy=True)
        search_k = min(k * 2, len(self.evidence_store))
        distances, indices = self.index.search(question_embedding.astype('float32'), search_k)
        year_match = re.search(r'20\d{2}', question)
        question_year = year_match.group(0) if year_match else None
        candidates = []
        for idx, dist in zip(indices[0], distances[0]):
            text, metadata = self.evidence_store[idx]
            score = float(dist)
            doc_name = metadata.get('doc_name', '')
            if question_year and question_year in doc_name:
                score = score * 0.5
            candidates.append({
                'evidence_text': text,
                'metadata': metadata,
                'distance': score,
            })
        candidates.sort(key=lambda x: x['distance'])
        return candidates[:k]

rag = FinancialRAG()
rag.load_index('./rag/vector_index')

# Quick test
test = rag.retrieve('What is the FY2018 capital expenditure for 3M?', k=1)
print(f'\nTest: {test[0]["metadata"]["company"]} - {test[0]["metadata"]["doc_name"]}')
print(f'Evidence: {test[0]["evidence_text"][:150]}...')
print('\n✅ Cell 9a done — RAG loaded!')

✅ RAG files already exist
Loading embedding model: all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Index loaded: 189 vectors, 189 evidence chunks

Test: 3M - 3M_2022_10K
Evidence: 3M Company and Subsidiaries
Consolidated Statement of Income
Years ended December 31
(Millions, except per share amounts)
2022
2021
2020
Net sales
$
3...

✅ Cell 9a done — RAG loaded!


In [18]:
# =====================================================================
# CELL 9b: Llama 3.1 8B + RAG — Split by Question Type
# =====================================================================

# ─── Step 1: Categorize all questions ────────────────────────

def categorize_question(gold_answer):
    gold_clean = gold_answer.strip().lower()
    if gold_clean.startswith('yes') or gold_clean.startswith('no'):
        return 'yes_no'
    words = gold_answer.strip().split()
    if len(words) <= 4:
        nums = extract_all_numbers(gold_answer)
        if nums:
            return 'numerical'
    return 'descriptive'

for ex in qa:
    ex['category'] = categorize_question(ex['answer'])

cat_counts = {}
for ex in qa:
    cat_counts[ex['category']] = cat_counts.get(ex['category'], 0) + 1

print(f'Question Categories:')
print(f'  Numerical:   {cat_counts.get("numerical", 0)}')
print(f'  Yes/No:      {cat_counts.get("yes_no", 0)}')
print(f'  Descriptive: {cat_counts.get("descriptive", 0)}')
print(f'  Total:       {len(qa)}')

# ─── Step 2: Different prompts per category ──────────────────

def build_rag_prompt_numerical(question, evidence_text):
    return f"""You are a financial data extraction tool.
Extract ONLY the specific number or value asked for from the evidence below.
Do NOT explain. Do NOT repeat the question. Do NOT write sentences.
Output format: just the number with units (e.g. "$1,577.0M" or "5.1%")

Evidence:
{evidence_text}

Question: {question}

Answer (number only):"""

def build_rag_prompt_yesno(question, evidence_text):
    return f"""You are a financial analyst. Based ONLY on the evidence below, answer the question.
Start with "Yes" or "No", then give a brief explanation with key numbers from the evidence.

Evidence:
{evidence_text}

Question: {question}

Answer (start with Yes or No):"""

def build_rag_prompt_descriptive(question, evidence_text):
    return f"""You are a financial analyst. Based ONLY on the evidence below, answer the question.
Give a concise answer using specific numbers and facts from the evidence.
Cite the source document when possible.

Evidence:
{evidence_text}

Question: {question}

Answer:"""

# ─── Step 3: Run evaluation ─────────────────────────────────

print(f'\n{"=" * 70}')
print(f'Llama 3.1 8B + RAG EVALUATION — {len(qa)} Questions')
print(f'{"=" * 70}')

correct_rag = 0
results_rag = []

for i, ex in enumerate(qa):
    q = ex['question']
    gold = ex['answer']
    fb_id = ex['financebench_id']
    evidence = ex.get('evidence', [])
    category = ex['category']

    # RAG: Retrieve evidence
    retrieved = rag.retrieve(q, k=3)
    evidence_text = '\n\n'.join([r['evidence_text'][:800] for r in retrieved])

    # Pick the right prompt for this question type
    if category == 'numerical':
        prompt = build_rag_prompt_numerical(q, evidence_text)
    elif category == 'yes_no':
        prompt = build_rag_prompt_yesno(q, evidence_text)
    else:
        prompt = build_rag_prompt_descriptive(q, evidence_text)

    # Generate
    try:
        pred_raw = generate_answer(prompt)
        pred = extract_answer_from_verbose(pred_raw) if category == 'numerical' else pred_raw
    except Exception as e:
        print(f'  ⚠️ Error Q{i+1}: {e}')
        pred_raw = pred = 'ERROR'

    # Correctness check
    ok = is_correct(pred, gold)
    correct_rag += int(ok)

    # Store all info for split scoring later
    results_rag.append({
        'financebench_id': fb_id,
        'company': ex['company'],
        'doc_name': ex['doc_name'],
        'question': q,
        'gold_answer': gold,
        'pred_raw': pred_raw,
        'pred_answer': pred,
        'correct': ok,
        'category': category,
    })

    if (i + 1) % 10 == 0:
        acc_so_far = correct_rag / (i + 1) * 100
        print(f'  [{i+1:>3}/{len(qa)}] Accuracy: {acc_so_far:.1f}%')

    # Show first 2 of each category
    shown = {'numerical': 0, 'yes_no': 0, 'descriptive': 0}
    for r in results_rag:
        if r['category'] == category and shown[category] < 2:
            if i < 6:
                print(f'\n  [{category.upper()}] Q: {q[:65]}...')
                print(f'    Gold: {gold[:70]}')
                print(f'    Pred: {pred_raw[:70]}')
                print(f'    Correct: {"✅" if ok else "❌"}')
            shown[category] += 1

print(f'\n  Overall Accuracy: {correct_rag}/{len(qa)} ({correct_rag/len(qa)*100:.1f}%)')


# ─── Step 4: Split Scoring ───────────────────────────────────

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

sem_model = SentenceTransformer('all-MiniLM-L6-v2')

# NUMERICAL
num_r = [r for r in results_rag if r['category'] == 'numerical']
num_acc_list = [numerical_accuracy(r['pred_answer'], r['gold_answer']) for r in num_r]
num_hal_list = [numerical_hallucination_rate(r['pred_answer'], evidence_map.get(r['financebench_id'], [])) for r in num_r]
num_cit_list = [has_citation(r['pred_raw']) for r in num_r]

# YES/NO
yn_r = [r for r in results_rag if r['category'] == 'yes_no']
yn_correct = 0
yn_bleu_list = []
yn_sem_list = []
yn_cit_list = []
for r in yn_r:
    g = r['gold_answer'].strip().lower()
    p = r['pred_answer'].strip().lower()
    if (g.startswith('yes') and p.startswith('yes')) or (g.startswith('no') and p.startswith('no')):
        yn_correct += 1
    yn_bleu_list.append(calculate_bleu(r['gold_answer'], r['pred_answer']))
    emb = sem_model.encode([r['gold_answer'], r['pred_answer']])
    yn_sem_list.append(float(cosine_similarity([emb[0]], [emb[1]])[0][0]))
    yn_cit_list.append(has_citation(r['pred_raw']))

# DESCRIPTIVE
desc_r = [r for r in results_rag if r['category'] == 'descriptive']
desc_bleu_list = []
desc_sem_list = []
desc_keynum_list = []
desc_cit_list = []
for r in desc_r:
    desc_bleu_list.append(calculate_bleu(r['gold_answer'], r['pred_answer']))
    emb = sem_model.encode([r['gold_answer'], r['pred_answer']])
    desc_sem_list.append(float(cosine_similarity([emb[0]], [emb[1]])[0][0]))
    gold_nums = extract_all_numbers(r['gold_answer'])
    pred_nums = extract_all_numbers(r['pred_answer'])
    if gold_nums:
        matched = 0
        for g in gold_nums:
            for p in pred_nums:
                if g != 0 and abs(p - g) / abs(g) < 0.05:
                    matched += 1; break
                elif g == 0 and abs(p) < 1e-6:
                    matched += 1; break
        desc_keynum_list.append(matched / len(gold_nums))
    else:
        desc_keynum_list.append(1.0)
    desc_cit_list.append(has_citation(r['pred_raw']))


# ─── Step 5: Print Results ───────────────────────────────────

print(f'\n{"=" * 70}')
print(f'  Llama 3.1 8B + RAG — SPLIT RESULTS')
print(f'{"=" * 70}')

print(f'\n  📊 NUMERICAL ({len(num_r)} questions)')
print(f'  {"─" * 50}')
print(f'  Numerical Accuracy:      {np.mean(num_acc_list)*100:.1f}%  ({int(sum(num_acc_list))}/{len(num_r)})')
print(f'  Numerical Hallucination: {np.mean(num_hal_list)*100:.1f}%')
print(f'  Citation Rate:           {np.mean(num_cit_list)*100:.1f}%')

print(f'\n  ✅❌ YES/NO ({len(yn_r)} questions)')
print(f'  {"─" * 50}')
print(f'  Yes/No Accuracy:         {yn_correct/len(yn_r)*100:.1f}%  ({yn_correct}/{len(yn_r)})')
print(f'  BLEU Score:              {np.mean(yn_bleu_list):.3f}')
print(f'  Semantic Similarity:     {np.mean(yn_sem_list)*100:.1f}%')
print(f'  Citation Rate:           {np.mean(yn_cit_list)*100:.1f}%')

print(f'\n  📝 DESCRIPTIVE ({len(desc_r)} questions)')
print(f'  {"─" * 50}')
print(f'  BLEU Score:              {np.mean(desc_bleu_list):.3f}')
print(f'  Semantic Similarity:     {np.mean(desc_sem_list)*100:.1f}%')
print(f'  Key Number Capture:      {np.mean(desc_keynum_list)*100:.1f}%')
print(f'  Citation Rate:           {np.mean(desc_cit_list)*100:.1f}%')

print(f'\n{"=" * 70}')

# Save
with open('results/llama_8b_rag_detailed.json', 'w') as f:
    json.dump(results_rag, f, indent=2)

with open('results/llama_8b_rag_split_metrics.json', 'w') as f:
    json.dump({
        'model': 'Llama-3.1-8B + RAG',
        'numerical': {
            'count': len(num_r),
            'accuracy': round(np.mean(num_acc_list)*100, 1),
            'hallucination': round(np.mean(num_hal_list)*100, 1),
            'citation': round(np.mean(num_cit_list)*100, 1),
        },
        'yes_no': {
            'count': len(yn_r),
            'yn_accuracy': round(yn_correct/len(yn_r)*100, 1),
            'bleu': round(np.mean(yn_bleu_list), 3),
            'semantic_similarity': round(np.mean(yn_sem_list)*100, 1),
            'citation': round(np.mean(yn_cit_list)*100, 1),
        },
        'descriptive': {
            'count': len(desc_r),
            'bleu': round(np.mean(desc_bleu_list), 3),
            'semantic_similarity': round(np.mean(desc_sem_list)*100, 1),
            'key_number_capture': round(np.mean(desc_keynum_list)*100, 1),
            'citation': round(np.mean(desc_cit_list)*100, 1),
        }
    }, f, indent=2)

print(f'\n✅ Cell 9b done!')
print(f'📌 Next: Run Cell 10 for BASE vs RAG comparison')

Question Categories:
  Numerical:   53
  Yes/No:      37
  Descriptive: 60
  Total:       150

Llama 3.1 8B + RAG EVALUATION — 150 Questions

  [NUMERICAL] Q: What is the FY2018 capital expenditure amount (in USD millions) f...
    Gold: $1577.00
    Pred: $1,444.0M

Question: What is the FY2022 total current assets amount (i
    Correct: ❌

  [NUMERICAL] Q: Assume that you are a public equities analyst. Answer the followi...
    Gold: $8.70
    Pred: $8.8B
Evidence:
Table of Property, Plant and Equipment, Net (PPNE) for
    Correct: ✅

  [NUMERICAL] Q: Assume that you are a public equities analyst. Answer the followi...
    Gold: $8.70
    Pred: $8.8B
Evidence:
Table of Property, Plant and Equipment, Net (PPNE) for
    Correct: ✅

  [YES_NO] Q: Is 3M a capital-intensive business based on FY2022 data?...
    Gold: No, the company is managing its CAPEX and Fixed Assets pretty efficien
    Pred: No

Explanation: 3M's current ratio is 0.65 (Current Assets / Current 
    Correct: ❌

  [DES

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.



  Llama 3.1 8B + RAG — SPLIT RESULTS

  📊 NUMERICAL (53 questions)
  ──────────────────────────────────────────────────
  Numerical Accuracy:      17.0%  (9/53)
  Numerical Hallucination: 22.6%
  Citation Rate:           54.7%

  ✅❌ YES/NO (37 questions)
  ──────────────────────────────────────────────────
  Yes/No Accuracy:         56.8%  (21/37)
  BLEU Score:              0.018
  Semantic Similarity:     62.8%
  Citation Rate:           73.0%

  📝 DESCRIPTIVE (60 questions)
  ──────────────────────────────────────────────────
  BLEU Score:              0.041
  Semantic Similarity:     61.9%
  Key Number Capture:      68.8%
  Citation Rate:           91.7%


✅ Cell 9b done!
📌 Next: Run Cell 10 for BASE vs RAG comparison
